# Assignment 4 — Cognitive Computing (UCS420)
## A Cognitive FAQ System Using Pandas (Nova 2.0)

**Name:** Jasmandeep Sharma  
**Roll Number:** 1024170269

## Q1. Build a personalized knowledge base

In [ ]:
import pandas as pd

fixed_entries = [
    {'question': 'what is the annual fee',
     'answer': 'The annual fee is Rs 500.',
     'keywords': 'fee cost price charge', 'category': 'billing'},
    {'question': 'how to reset password',
     'answer': 'Go to Settings > Reset Password.',
     'keywords': 'password reset login', 'category': 'account'},
    {'question': 'what are your working hours',
     'answer': 'We are open 9 AM to 5 PM.',
     'keywords': 'hours timing open time', 'category': 'general'},
    {'question': 'how can i pay the fee',
     'answer': 'You can pay via UPI, card, or net banking.',
     'keywords': 'pay payment upi fee', 'category': 'billing'},
]

# Last two digits of roll number: 6 and 9
roll_number = '1024170269'
categories = ['billing', 'account', 'general']
personalized_entries = []

for digit in roll_number[-2:]:
    category = categories[int(digit) % 3]
    if category == 'billing':
        entry = {
            'question': 'how can I check the status of my fee payment',
            'answer': 'Check the payment history in your account or contact the billing office.',
            'keywords': 'payment status transaction fee',
            'category': category
        }
    elif category == 'account':
        entry = {
            'question': 'how do I update my registered mobile number',
            'answer': 'Open account settings, update your mobile number, and verify it using the OTP.',
            'keywords': 'mobile number update account otp',
            'category': category
        }
    else:
        entry = {
            'question': 'where can I get help with a general query',
            'answer': 'Contact the help desk during working hours for assistance.',
            'keywords': 'help desk support query',
            'category': category
        }
    personalized_entries.append(entry)

faq_df = pd.DataFrame(fixed_entries + personalized_entries)
print('Final 6-row FAQ DataFrame:')
display(faq_df)

## Q2. Generate and score a hypothesis

In [ ]:
def score_query(query, df):
    """Score FAQ entries by the number of query words found in their text."""
    query_words = set(query.lower().split())
    results = []
    for index, row in df.iterrows():
        searchable_text = ' '.join([
            str(row['question']), str(row['answer']),
            str(row['keywords']), str(row['category'])
        ]).lower()
        score = sum(1 for word in query_words if word in searchable_text)
        if score > 0:
            results.append({'index': index, 'score': score,
                            'question': row['question'], 'answer': row['answer'],
                            'category': row['category']})
    results.sort(key=lambda item: item['score'], reverse=True)
    return pd.DataFrame(results)

query = 'reset password'
ranked_results = score_query(query, faq_df)
display(ranked_results)

## Q3. Return all questions from a category

In [ ]:
def same_category(category_name, df):
    return df.loc[df['category'].str.lower() == category_name.lower(), ['question', 'answer', 'keywords', 'category']]

# Call the function using the category of a personalized entry
personalized_category = personalized_entries[0]['category']
print('Category:', personalized_category)
display(same_category(personalized_category, faq_df))

## Q4. Add a keyword and save the updated DataFrame

In [ ]:
# Choose the first entry, ask for one new keyword, and append it.
entry_index = 0
new_keyword = input('Enter a new keyword to add to the first FAQ entry: ').strip()
if new_keyword:
    existing_keywords = faq_df.at[entry_index, 'keywords']
    faq_df.at[entry_index, 'keywords'] = f'{existing_keywords} {new_keyword}'
    print('Updated keywords:', faq_df.at[entry_index, 'keywords'])
else:
    print('No keyword entered; keywords were not changed.')

output_file = f'{roll_number}_faq_data.csv'
faq_df.to_csv(output_file, index=False)
print(f'Saved updated FAQ data to {output_file}')

## Q5. Count FAQ entries per category

In [ ]:
faq_df.groupby('category').size().reset_index(name='FAQ_Count')

## Q6. Display all top-scoring ties

In [ ]:
def score_query_with_ties(query, df):
    """Return every highest-scoring match and report if the top score is tied."""
    query_words = set(query.lower().split())
    results = []
    for index, row in df.iterrows():
        searchable_text = ' '.join([
            str(row['question']), str(row['answer']),
            str(row['keywords']), str(row['category'])
        ]).lower()
        score = sum(1 for word in query_words if word in searchable_text)
        if score > 0:
            results.append({'index': index, 'score': score,
                            'question': row['question'], 'answer': row['answer'],
                            'category': row['category']})
    if not results:
        print('No matching FAQ entries found.')
        return pd.DataFrame()
    highest_score = max(item['score'] for item in results)
    top_matches = [item for item in results if item['score'] == highest_score]
    if len(top_matches) > 1:
        print(f'Tie: {len(top_matches)} entries share the highest score ({highest_score}).')
    else:
        print(f'One highest-scoring match (score {highest_score}).')
    return pd.DataFrame(top_matches)

# Demonstration 1: query intended to match both fee-related entries
print('Tie demonstration:')
display(score_query_with_ties('fee', faq_df))

# Demonstration 2: query with a single clear top match
print('Non-tie demonstration:')
display(score_query_with_ties('password reset login', faq_df))